Use web_search
```
        {
            "id": "srvtoolu_01N5nMsvQq8Gj3XCDx4sSzn2",
            "caller": null,
            "input": {
                "query": "New York weather today"
            },
            "name": "web_search",
            "type": "server_tool_use"
        }

```

```
            "tool_use_id": "srvtoolu_01N5nMsvQq8Gj3XCDx4sSzn2",
            "type": "web_search_tool_result"

```

In [2]:
from util.claude_client import get_claude_client_and_default_model

client , llm= get_claude_client_and_default_model('claude' )

web_tools = [
    {"type": "web_search_20250305", "name": "web_search", "max_uses": 2},
    {"type": "web_fetch_20250910", "name": "web_fetch", "max_uses": 2},
]

message = client.messages.create(
    max_tokens = 1024,
    model=llm,
    messages= [ {
          'role' : 'user',
          'content': 'What is the weather of New York'
        }
    ],
    tools = web_tools
)

print( message.model_dump_json(indent=4) )

llm profile name : claude , model : claude-haiku-4-5-20251001
{
    "id": "msg_011CfVzwjHqjkExVDefnT6uQ",
    "container": null,
    "content": [
        {
            "citations": null,
            "text": "I'll search for the current weather in New York for you.",
            "type": "text"
        },
        {
            "id": "srvtoolu_01N5nMsvQq8Gj3XCDx4sSzn2",
            "caller": null,
            "input": {
                "query": "New York weather today"
            },
            "name": "web_search",
            "type": "server_tool_use"
        },
        {
            "caller": {
                "type": "direct"
            },
            "content": [
                {
                    "encrypted_content": "EuAQCioIFBgCIiQxNWM0MzVlYy03MmVmLTRkMzQtOWVmOS0zZjM2MWQ3MTQxOTMSDL/hI6W+4AhjejdzKxoMjIRuXJEXBs9/k8XyIjDDWjcgBgvRVBkNFj+e7xv0REvxweOg7DWHYy6libpRE8DCC1oryV0k9xf47no+SwAq4w8+BfAV0xn4DeAehBIrIa5f8ijscLaI+lSrWkhfYKwOfFVya+bEHY/zC+LbzvMhcJz/PPo9sXiJdhyYm7XC8TmbGyVFc6Sb

When both client tool and server tool are enabled
It calls get_weather , web_search is skipped

In [4]:
weather_tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather in a given location",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                }
            },
            "required": ["location"],
        },
    }
]

message = client.messages.create(
    max_tokens = 1024,
    model=llm,
    messages= [ {
          'role' : 'user',
          'content': 'What is the weather of New York'
        }
    ],
    tools = web_tools + weather_tools,
    tool_choice={"type": "auto", "disable_parallel_tool_use": True}

)

print( message.model_dump_json(indent=4) )


{
    "id": "msg_011CfW1Yo8kadLsuFhbjgoN8",
    "container": null,
    "content": [
        {
            "id": "toolu_01TfT8qBx9eEXNtN7P2H5im8",
            "caller": {
                "type": "direct"
            },
            "input": {
                "location": "New York, NY"
            },
            "name": "get_weather",
            "type": "tool_use",
            "toolset_name": null
        }
    ],
    "model": "claude-haiku-4-5-20251001",
    "role": "assistant",
    "stop_details": null,
    "stop_reason": "tool_use",
    "stop_sequence": null,
    "type": "message",
    "usage": {
        "cache_creation": {
            "ephemeral_1h_input_tokens": 0,
            "ephemeral_5m_input_tokens": 0
        },
        "cache_creation_input_tokens": 0,
        "cache_read_input_tokens": 0,
        "inference_geo": "not_available",
        "input_tokens": 2611,
        "output_tokens": 48,
        "output_tokens_details": null,
        "server_tool_use": null,
        "service

use tool_choice to force web_search
```
tool_choice: {"type": "tool", "name": "web_search"}
```
then client_tool get_weather is not selected

In [5]:
message = client.messages.create(
    max_tokens = 1024,
    model=llm,
    messages= [ {
          'role' : 'user',
          'content': 'What is the weather of New York'
        }
    ],
    tools = web_tools + weather_tools,
    tool_choice={"type": "tool", "name": "web_search"}

)

print( message.model_dump_json(indent=4) )

{
    "id": "msg_011CfW1iLhYYcp8eL56qHEtg",
    "container": null,
    "content": [
        {
            "id": "srvtoolu_016eK6jMhEFJpbpejD3HT3je",
            "caller": null,
            "input": {
                "query": "weather New York today"
            },
            "name": "web_search",
            "type": "server_tool_use"
        },
        {
            "caller": {
                "type": "direct"
            },
            "content": [
                {
                    "encrypted_content": "EuAQCioIFBgCIiQxNWM0MzVlYy03MmVmLTRkMzQtOWVmOS0zZjM2MWQ3MTQxOTMSDHVdTtUKpqRHRZxJgxoMuxI5mT7XBabUvweVIjBWPS1x7sXyZrQeq+Eqg7HccdA9kfep5m5gX7fEVzACr+cLGPkamjQR20PqoSLCClQq4w/0n8GFCV51FgX89vPFc7SYWOcvS7fqlHfavP+/8+iff/+QAKUhCyRrg68qOc1ufGlmkIpk0nAktXdzKE4ayec5ecqueH8tcQu1wUFbFRbhAASI4Ci0NZNNRjhlKRDrhRfioimS2k32/8XFfihIlZZDqU5cMCvGkdqqC9IkLjXEGd4KdPVXIII2axbk6Z9+j0KQOp0z8oZedJOrywaRUZSBdZlJwcaK/rnWDvxteA/+FHzo4Kdh5UJYhNvG5B9hHmOS6ACsc6ygKg78dDZwXp9AHiZrYBVP02xAMZ0LAN09WlldNCDeXk6FkOh0M